### Dataset and Task Metadata

In [2]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="drug_induced_autoimmunity_prediction",
    dataset_year="2025",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5332M",
    download_description="""
wget https://archive.ics.uci.edu/static/public/1104/drug_induced_autoimmunity_prediction.zip && unzip drug_induced_autoimmunity_prediction.zip && rm drug_induced_autoimmunity_prediction.zip RDKit_ChemDes.xlsx && mkdir -p local-data-warehouse/drug_induced_autoimmunity_prediction && mv DIA_trainingset_RDKit_descriptors.csv local-data-warehouse/drug_induced_autoimmunity_prediction/ && mv DIA_testset_RDKit_descriptors.csv local-data-warehouse/drug_induced_autoimmunity_prediction/
""",
    # References
    academic_reference_bibtex="""@article{huang2025interdia,
  title={InterDIA: Interpretable prediction of drug-induced autoimmunity through ensemble machine learning approaches},
  author={Huang, Lina and Liu, Peineng and Huang, Xiaojie},
  journal={Toxicology},
  volume={511},
  pages={154064},
  year={2025},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="huang2025interdia",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the dataset from UCI.

- We drop the constant columns.
- We keep the SMILES code as string for later pipelines to handle.

""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Label",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Label",
)

## Preprocessing

In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "DIA_trainingset_RDKit_descriptors.csv")
df = pd.concat([df, pd.read_csv(dataset_mold.path / "DIA_testset_RDKit_descriptors.csv")], ignore_index=True)
print("Loaded data shape:", df.shape)

# Drop constant columns
constant_cols = df.columns[df.nunique(dropna=False) == 1]
df = df.drop(columns=constant_cols)

# dropping duplicated columns (by chance)
duplicated = ["fr_Nhpyrrole", "MaxEStateIndex", "fr_benzene"]
df = df.drop(columns=duplicated)

as_string_type = ["SMILES"]
as_cat_type = ["Label"]

for c in as_string_type:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")
df[as_cat_type] = df[as_cat_type].astype("category")


df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (597, 198)


## Data Checks

In [11]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 597
Columns: 178
Use sampling: False (sample size: 597)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['SMILES', 'MolMR', 'Chi1v', 'Chi2v', 'LabuteASA', 'BertzCT', 'Chi0v', 'MolWt', 'Chi1n', 'ExactMolWt']
Rows remaining as candidates after top-10 filter: 0 (of 597)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [12]:
# Sample Rows
df_head

,Label,SMILES,BalabanJ,BertzCT,Chi0,Chi0n,Chi0v,Chi1,Chi1n,Chi1v,Chi2n,Chi2v,Chi3n,Chi3v,Chi4n,Chi4v,EState_VSA1,EState_VSA10,EState_VSA11,EState_VSA2,EState_VSA3,EState_VSA4,EState_VSA5,EState_VSA6,EState_VSA7,EState_VSA8,EState_VSA9,ExactMolWt,FractionCSP3,HallKierAlpha,HeavyAtomCount,HeavyAtomMolWt,Ipc,Kappa1,Kappa2,Kappa3,LabuteASA,MaxAbsEStateIndex,MaxAbsPartialCharge,MaxPartialCharge,MinAbsEStateIndex,MinAbsPartialCharge,MinEStateIndex,MinPartialCharge,MolLogP,MolMR,MolWt,NHOHCount,NOCount,NumAliphaticCarbocycles,NumAliphaticHeterocycles,NumAliphaticRings,NumAromaticCarbocycles,NumAromaticHeterocycles,NumAromaticRings,NumHAcceptors,NumHDonors,NumHeteroatoms,NumRotatableBonds,NumSaturatedCarbocycles,NumSaturatedHeterocycles,NumSaturatedRings,NumValenceElectrons,PEOE_VSA1,PEOE_VSA10,PEOE_VSA11,PEOE_VSA12,PEOE_VSA13,PEOE_VSA14,PEOE_VSA2,PEOE_VSA3,PEOE_VSA4,PEOE_VSA5,PEOE_VSA6,PEOE_VSA7,PEOE_VSA8,PEOE_VSA9,RingCount,SMR_VSA1,SMR_VSA10,SMR_VSA2,SMR_VSA3,SMR_VSA4,SMR_VSA5,SMR_VSA6,SMR_VSA7,SMR_VSA9,SlogP_VSA1,SlogP_VSA10,SlogP_VSA11,SlogP_VSA12,SlogP_VSA2,SlogP_VSA3,SlogP_VSA4,SlogP_VSA5,SlogP_VSA6,SlogP_VSA7,SlogP_VSA8,TPSA,VSA_EState10,VSA_EState8,VSA_EState9,fr_Al_COO,fr_Al_OH,fr_Al_OH_noTert,fr_ArN,fr_Ar_COO,fr_Ar_N,fr_Ar_NH,fr_Ar_OH,fr_COO,fr_COO2,fr_C_O,fr_C_O_noCOO,fr_C_S,fr_HOCCN,fr_Imine,fr_NH0,fr_NH1,fr_NH2,fr_N_O,fr_Ndealkylation1,fr_Ndealkylation2,fr_SH,fr_aldehyde,fr_alkyl_carbamate,fr_alkyl_halide,fr_allylic_oxid,fr_amide,fr_amidine,fr_aniline,fr_aryl_methyl,fr_azo,fr_benzodiazepine,fr_bicyclic,fr_dihydropyridine,fr_epoxide,fr_ester,fr_ether,fr_furan,fr_guanido,fr_halogen,fr_hdrzine,fr_hdrzone,fr_imidazole,fr_imide,fr_ketone,fr_ketone_Topliss,fr_lactam,fr_lactone,fr_methoxy,fr_morpholine,fr_nitrile,fr_nitro,fr_nitro_arom,fr_nitro_arom_nonortho,fr_nitroso,fr_oxazole,fr_oxime,fr_para_hydroxylation,fr_phenol,fr_phenol_noOrthoHbond,fr_phos_acid,fr_phos_ester,fr_piperdine,fr_piperzine,fr_priamide,fr_pyridine,fr_quatN,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiophene,fr_unbrch_alkane,fr_urea
0,1,OCCN1CCN(CCCN2c3ccccc3Sc4ccc(Cl)cc24)CC1,1.410,779.772,18.640,15.480,17.053,13.242,9.640,10.834,7.094,8.614,5.325,6.690,3.874,5.116,0.000,0.000,0.0,6.607,0.000,57.257,21.166,0.000,17.828,51.098,16.707,403.149,0.429,-1.24,27,377.771,2100937.376,19.093,8.842,4.438,170.261,9.078,0.395,0.057,0.258,0.057,0.258,-0.395,3.943,113.606,403.979,1,4,0,2,2,2,0,2,5,1,6,6,0,1,1,144,14.906,0.000,0.000,0.000,0.0,0.000,4.900,0.000,0.0,0.0,35.496,43.297,54.082,17.982,4,5.107,34.738,0.0,9.8,0.000,16.212,57.320,47.487,0.0,4.900,11.375,0.0,23.363,67.327,0.000,0.000,6.421,52.256,5.023,0.0,29.95,8.122,0.000,45.656,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,3,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,2,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
1,1,NC(=O)Cc1cccc(C(=O)c2ccccc2)c1N,2.406,621.298,13.828,10.297,10.297,9.092,5.847,5.847,4.217,4.217,2.842,2.842,1.898,1.898,5.907,9.589,0.0,12.204,22.378,0.000,0.000,42.465,6.066,0.000,11.467,254.106,0.067,-2.62,19,240.177,20828.746,12.827,5.351,2.796,110.586,12.309,0.398,0.221,0.030,0.221,-0.476,-0.398,1.528,73.627,254.289,4,4,0,0,0,2,0,2,3,2,4,4,0,0,0,96,11.467,0.000,5.783,5.907,0.0,0.000,9.589,0.000,0.0,0.0,42.465,11.630,16.814,6.421,2,9.589,17.378,0.0,0.0,5.734,6.421,5.734,65.221,0.0,11.467,5.687,0.0,0.000,11.690,11.215,0.000,21.485,48.531,0.000,0.0,86.18,0.000,0.000,49.500,0,0,0,1,0,0,0,0,0,0,2,2,0,0,0,0,0,2,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
2,1,OC(=O)[C@@H]1C[C@H](CN1C(=O)CP(=O)(O)CCCCc2ccccc2)C3CCCCC3,1.501,762.117,21.562,17.618,18.512,14.355,11.226,13.256,8.965,11.421,6.703,8.483,5.039,6.519,31.449,24.154,0.0,12.080,25.304,25.683,29.726,0.000,30.332,0.000,0.000,435.217,0.652,-1.65,30,401.229,5910365.206,23.022,10.588,6.422,178.671,12.789,0.480,0.326,0.100,0.326,-3.615,-0.480,4.162,116.758,435.501,2,6,1,1,2,1,0,1,3,2,7,10,1,1,2,166,14.900,12.204,0.000,13.276,0.0,5

In [13]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Label,category,0.0,0.0,2.0,"0, 1"
1,BalabanJ,float64,0.0,0.0,487.0,"1.838, 1.735, 2.195, 2.462, 2.218, 2.079, 1.85, 1.43, 1.384, 1.797"
2,BertzCT,float64,0.0,0.0,575.0,"682.154, 983.598, 400.94, 593.895, 49.784, 431.802, 760.387, 666.182, 1053.003, 804.749"
3,Chi0,float64,0.0,0.0,449.0,"10.836, 20.698, 13.828, 12.345, 22.422, 16.397, 11.707, 12.958, 23.859, 15.267"
4,Chi0n,float64,0.0,0.0,570.0,"17.145, 16.613, 17.935, 14.202, 13.404, 19.777, 14.059, 4.957, 4.331, 18.174"
5,Chi0v,float64,0.0,0.0,571.0,"13.404, 16.626, 21.884, 20.593, 13.542, 14.059, 20.6, 14.202, 16.995, 18.174"
6,Chi1,float64,0.0,0.0,515.0,"12.963, 14.867, 4.877, 9.665, 4.5, 9.075, 14.918, 11.952, 4.164, 6.914"
7,Chi1n,float64,0.0,0.0,570.0,"4.609, 11.259, 1.982, 8.308, 8.683, 6.728, 9.823, 7.049, 2.745, 9.128"
8,Chi1v,float64,0.0,0.0,577.0,"12.377, 5.98, 9.298, 7.771, 12.729, 11.19, 3.683, 8.683, 9.128, 9.926"
9,Chi2n,float64,0.0,0.0,565.0,"7.662, 6.974, 7.441, 5.461, 3.781, 5.49, 8.304, 2.667, 5.397, 9.355"


In [14]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
BalabanJ,597.0,2.143109e+00,7.242450e-01,0.837,5.083000e+00
BertzCT,597.0,7.521650e+02,4.042390e+02,2.000,2.629395e+03
Chi0,597.0,1.844259e+01,7.352206e+00,2.000,5.012000e+01
Chi0n,597.0,1.463103e+01,6.153925e+00,0.908,4.132800e+01
Chi0v,597.0,1.513229e+01,6.204312e+00,0.908,4.132800e+01
Chi1,597.0,1.207100e+01,4.895135e+00,1.000,3.235000e+01
Chi1n,597.0,8.573367e+00,3.778853e+00,0.204,2.326900e+01
Chi1v,597.0,9.076570e+00,3.838328e+00,0.204,2.444000e+01
Chi2n,597.0,6.800184e+00,3.322090e+00,0.000,1.935200e+01
Chi2v,597.0,7.351312e+00,3.375425e+00,0.000,1.961500e+01


In [15]:
# Categorical Feature Statistics
cat_stats

value  \
column rank                                                                        
Label  1                                                                       0   
       2                                                                       1   
SMILES 1                                       CC(C)NC[C@H](O)COc1cccc2[nH]ccc12   
       2                                OCCN1CCN(CCCN2c3ccccc3Sc4ccc(Cl)cc24)CC1   
       3                                         NC(=O)Cc1cccc(C(=O)c2ccccc2)c1N   
       4              OC(=O)[C@@H]1C[C@H](CN1C(=O)CP(=O)(O)CCCCc2ccccc2)C3CCCCC3   
       5     CC[C@H](C)C(=O)O[C@@H]1C[C@H](C)C=C2C=C[C@H](C)[C@H](CC[C@@H]3C[...   

             count    pct  
column rank                
Label  1       449  75.21  
       2       148  24.79  
SMILES 1         1   0.17  
       2         1   0.17  
       3         1   0.17  
       4         1   0.17  
       5         1   0.17

In [16]:
# Target Distribution
target_df

,count,pct
Label,,
0,449,75.21
1,148,24.79


## Task Curation

In [17]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [18]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [19]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c9f82-d8aa-7c13-b593-d70835a80e0a
614393f00340d2a0e9ffcecd39bfacabdbc16c461ab886004a0bda1159c7d0b6
